# MAE 6246 - Week 1 Python Example

## Nonlinear pendulum and local linearizations

This notebook accompanies the Week 1 note on state-space models and linearization. We will:

1. simulate the nonlinear torque-driven pendulum;
2. construct Jacobian linearizations about the downward and upright equilibria;
3. compare each local model with the nonlinear response; and
4. quantify how approximation error changes with the initial perturbation.

The angle convention is $\theta=0$ downward and $\theta=\pi$ upright. All simulations below use zero applied torque.

## 1. Model

For a point mass $m$ on a massless rod of length $\ell$,

$$m\ell^2\ddot\theta=-mg\ell\sin\theta+u.$$

With $x=[\theta,\dot\theta]^T$,

$$\dot x=f(x,u)=\begin{bmatrix}x_2\\-(g/\ell)\sin x_1+u/(m\ell^2)\end{bmatrix}.$$

At the equilibrium $x^\star=[\theta_{eq},0]^T$, the Jacobian is

$$A(\theta_{eq})=\begin{bmatrix}0&1\\-(g/\ell)\cos\theta_{eq}&0\end{bmatrix}.$$

The linear state is the perturbation $\delta x=x-x^\star$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

g = 9.81       # gravitational acceleration [m/s^2]
ell = 1.0      # pendulum length [m]
m = 1.0        # point mass [kg]

In [ ]:
def pendulum_rhs(t, x, torque=lambda t: 0.0):
    """Nonlinear state equation with x = [theta, theta_dot]."""
    theta, theta_dot = x
    u = torque(t)
    return np.array([
        theta_dot,
        -(g / ell) * np.sin(theta) + u / (m * ell**2),
    ])


def linear_matrix(theta_eq):
    """Return the Jacobian A evaluated at [theta_eq, 0]."""
    return np.array([
        [0.0, 1.0],
        [-(g / ell) * np.cos(theta_eq), 0.0],
    ])


def compare_models(theta_eq, delta_theta0, t_final=6.0):
    """Simulate the nonlinear and local perturbation models."""
    t_eval = np.linspace(0.0, t_final, 1001)

    # The nonlinear model uses the physical angle theta.
    x0 = np.array([theta_eq + delta_theta0, 0.0])
    nonlinear = solve_ivp(
        pendulum_rhs,
        (0.0, t_final),
        x0,
        t_eval=t_eval,
        rtol=1e-10,
        atol=1e-12,
    )

    # The linearized model uses delta_x = x - x_star.
    A = linear_matrix(theta_eq)
    delta_x0 = np.array([delta_theta0, 0.0])
    linear = solve_ivp(
        lambda t, delta_x: A @ delta_x,
        (0.0, t_final),
        delta_x0,
        t_eval=t_eval,
        rtol=1e-10,
        atol=1e-12,
    )

    theta_nonlinear = nonlinear.y[0]
    theta_linear = theta_eq + linear.y[0]
    return t_eval, theta_nonlinear, theta_linear

## 2. Downward equilibrium

At $\theta_{eq}=0$,

$$A_d=\begin{bmatrix}0&1\\-g/\ell&0\end{bmatrix}.$$

Compare initial displacements of 5, 30, and 90 degrees. Before running the cell, predict which response will remain closest to its linear approximation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

for ax, angle_deg in zip(axes, [5, 30, 90]):
    t, theta_nl, theta_lin = compare_models(
        theta_eq=0.0,
        delta_theta0=np.deg2rad(angle_deg),
    )
    ax.plot(t, np.rad2deg(theta_nl), label="nonlinear", lw=2)
    ax.plot(t, np.rad2deg(theta_lin), "--", label="linearized", lw=2)
    ax.set_title(f"initial angle = {angle_deg} deg")
    ax.set_xlabel("time [s]")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("angle [deg]")
axes[0].legend()
fig.suptitle("Linearization about the downward equilibrium")
fig.tight_layout()
plt.show()

### Interpretation

- At 5 degrees, the curves nearly coincide.
- At 30 degrees, a phase error becomes visible.
- At 90 degrees, the linear model is qualitatively suggestive but quantitatively poor.

The small-angle approximation has leading error

$$\sin\theta-\theta\approx-\frac{\theta^3}{6},$$

so a tenfold decrease in a sufficiently small angle should reduce the instantaneous approximation error by roughly a factor of $10^3$.

## 3. Quantify the local-model error

For each initial angle, compute the maximum angle difference over three seconds. A log-log plot makes the small-perturbation scaling easier to see.

In [ ]:
initial_angles_deg = np.geomspace(0.1, 60.0, 30)
max_errors_deg = []

for angle_deg in initial_angles_deg:
    t, theta_nl, theta_lin = compare_models(
        theta_eq=0.0,
        delta_theta0=np.deg2rad(angle_deg),
        t_final=3.0,
    )
    error = np.max(np.abs(theta_nl - theta_lin))
    max_errors_deg.append(np.rad2deg(error))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(initial_angles_deg, max_errors_deg, "o-")
ax.set_xlabel("initial angle [deg]")
ax.set_ylabel("maximum angle error [deg]")
ax.set_title("Error of the downward linearization over 3 seconds")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

There is no universal angle at which the linearization suddenly becomes invalid. Model validity depends on the acceptable error, time interval, input, and question being asked.

## 4. Upright equilibrium

At $\theta_{eq}=\pi$,

$$A_u=\begin{bmatrix}0&1\\+g/\ell&0\end{bmatrix}.$$

The plotted quantity is the perturbation $\delta\theta=\theta-\pi$, not the physical angle itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for angle_deg in [0.5, 2.0, 8.0]:
    t, theta_nl, theta_lin = compare_models(
        theta_eq=np.pi,
        delta_theta0=np.deg2rad(angle_deg),
        t_final=1.5,
    )
    ax.plot(
        t,
        np.rad2deg(theta_nl - np.pi),
        label=f"nonlinear, {angle_deg:g} deg",
        lw=2,
    )
    ax.plot(
        t,
        np.rad2deg(theta_lin - np.pi),
        "--",
        label=f"linearized, {angle_deg:g} deg",
        lw=1.5,
    )

ax.set_xlabel("time [s]")
ax.set_ylabel(r"perturbation $\theta-\pi$ [deg]")
ax.set_title("Linearization about the upright equilibrium")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

The nonlinear and local responses initially agree, but both move away from the upright equilibrium. Eventually, the local approximation fails because the state has left the neighborhood where the Jacobian was evaluated.

This experiment separates two ideas that will be formalized later:

- **local accuracy:** whether the linearized and nonlinear trajectories agree near the operating point;
- **stability:** whether trajectories that start near the operating point remain near it.

## 5. Student investigations

1. Choose a maximum acceptable angle error, such as 1 degree. Estimate the largest downward initial angle satisfying that tolerance over three seconds.
2. Change the pendulum length. Explain how the response time scale changes.
3. Add a constant torque. Determine the new equilibrium and modify the linearization accordingly.
4. Give the pendulum a nonzero initial angular velocity. Compare the region over which the local model remains accurate.
5. Explain why the upright linearization can be accurate initially even though the equilibrium is unstable.